# R-vine fitting and diagnostics

Fit one truncated R-vine, inspect structure-selection diagnostics, test
goodness of fit, and verify which conditional-sampling path is used at
prediction time.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from pyscarcopula._utils import pobs
from pyscarcopula import RVineCopula
from pyscarcopula.stattests import gof_test

## Data and fit parameters

In [2]:
TICKERS = [
    "BTC-USD", "ETH-USD", "BNB-USD",
    "ADA-USD", "XRP-USD", "DOGE-USD",
]
ROWS = slice(0, 250)
GIVEN_VARS = [0]
FIT = {
    "method": "scar-tm-ou",
    "truncation_level": 2,
    "given_vars": GIVEN_VARS,
}
SEED = 2029

DATA_DIR = Path("data") if Path("data").exists() else Path("..") / "data"
prices = pd.read_csv(DATA_DIR / "crypto_prices.csv", index_col=0, sep=";")
returns = np.log(prices[TICKERS] / prices[TICKERS].shift(1)).dropna().iloc[ROWS]
u = pobs(returns.to_numpy())

## Fit and structure diagnostics

In [3]:
vine = RVineCopula().fit(u, **FIT)

vine.summary()
fit_diagnostics = vine.fit_diagnostics
pd.Series({
    "target_given_vars": fit_diagnostics["target_given_vars"],
    "conditional_fit_supported": (
        fit_diagnostics["conditional_fit_supported"]
    ),
    "selected_mode": fit_diagnostics["selection"]["selected_mode"],
    "candidate_count": len(fit_diagnostics["selection"]["candidates"]),
    "reject_reason": fit_diagnostics["reject_reason"],
})

RVineCopula(d=6, T=250, criterion='aic')
  log_likelihood = 885.7529
  n_parameters   = 27
  AIC = -1717.5059   BIC = -1622.4264

Structure matrix (natural order, Czado 2019 Alg. 5.4; anti-diagonal = leaf peeled at each column):
[[5 5 5 5 5 5]
 [0 0 0 0 0 0]
 [2 2 1 1 0 0]
 [3 1 2 0 0 0]
 [1 3 0 0 0 0]
 [4 0 0 0 0 0]]

Edges (tree t, column col):
   t  col       pair           cond  family              rot  dyn_params                                       param     tau       logL
   0    0      (4,1)              -  GumbelCopula        180  kappa= 62.412, mu=  2.343, nu= 12.479                     0.673    192.432
   0    1      (3,1)              -  GumbelCopula        180  kappa= 14.629, mu=  1.636, nu=  6.004                     0.588    151.521
   0    2      (2,1)              -  GumbelCopula        180  kappa= 51.239, mu=  2.394, nu= 17.408                     0.647    184.824
   0    3      (1,0)              -  GumbelCopula        180  kappa= 19.084, mu=  2.278, nu=  7.254     

target_given_vars                   (0,)
conditional_fit_supported           True
selected_mode                given_first
candidate_count                        1
reject_reason                       None
dtype: object

## Goodness of fit and unconditional sampling

In [4]:
gof = gof_test(vine, u, to_pobs=False)
sample_u = vine.sample(2_000, rng=np.random.default_rng(SEED))

print("logL:", vine.log_likelihood())
print("AIC / BIC:", vine.aic, vine.bic)
print("GoF p-value:", gof.pvalue)
print("sample shape:", sample_u.shape)

logL: 885.7529404328221
AIC / BIC: -1717.5058808656443 -1622.4264360833636
GoF p-value: 0.941285038051757
sample shape: (2000, 6)


## Conditional prediction diagnostics

In [5]:
given = {GIVEN_VARS[0]: 0.1}
conditional_u, prediction_diagnostics = vine.predict(
    3_000,
    u=u,
    given=given,
    horizon="current",
    return_diagnostics=True,
    rng=np.random.default_rng(SEED + 1),
)

print(pd.Series({
    "conditional_method": prediction_diagnostics["conditional_method"],
    "dynamic_conditioning": prediction_diagnostics["dynamic_conditioning"],
    "matrix_rebuilt": prediction_diagnostics["matrix_rebuilt"],
    "updated_edges": len(prediction_diagnostics["updated_edges"]),
    "skipped_edges": len(prediction_diagnostics["skipped_edges"]),
}))
print("fixed coordinate:", np.unique(conditional_u[:, GIVEN_VARS[0]]))
pd.Series(conditional_u.mean(axis=0), index=TICKERS)

conditional_method      suffix
dynamic_conditioning    ignore
matrix_rebuilt            True
updated_edges                0
skipped_edges                0
dtype: object
fixed coordinate: [0.1]


BTC-USD     0.100000
ETH-USD     0.186926
BNB-USD     0.240623
ADA-USD     0.239740
XRP-USD     0.230471
DOGE-USD    0.141485
dtype: float64

If `conditional_method` is `dag_mcmc`, also inspect
`prediction_diagnostics["mcmc"]` for acceptance rates and warnings. A suffix
structure selected for `GIVEN_VARS` normally uses the exact `suffix` path.